# 🧪 Test Suy Diễn Base Model Qwen2.5-3B

Notebook này kiểm tra khả năng suy diễn (inference) của base model **Qwen2.5-3B** trước khi fine-tune.

**Nội dung:**
1. Kiểm tra môi trường (GPU, CUDA)
2. Tải model & tokenizer
3. Test sinh văn bản (text generation)
4. Test với prompt dạng meeting summary
5. Đo thời gian và bộ nhớ

## 1. Kiểm tra môi trường

In [ ]:
import sys
import torch

print(f"Python : {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.version.cuda}")
print(f"GPU    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Không có GPU'}")
print(f"VRAM   : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB" if torch.cuda.is_available() else "")

## 2. Tải Model & Tokenizer

In [ ]:
import sys
import os

# Thêm root project vào sys.path để import modules
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from modules.model_loader import load_model_and_tokenizer

In [ ]:
# Tải model với quantization 4-bit để tiết kiệm VRAM
# Đổi use_4bit=False nếu muốn tải full precision
model, tokenizer = load_model_and_tokenizer(
    use_4bit=True,
    torch_dtype="bfloat16",
    device_map="auto",
)

print(f"Model device : {model.device}")
print(f"Model dtype  : {model.dtype}")
print(f"Vocab size   : {tokenizer.vocab_size:,}")
print(f"Pad token    : {tokenizer.pad_token} (id={tokenizer.pad_token_id})")

## 3. Hàm tiện ích để sinh văn bản

In [ ]:
import time

def generate_text(
    prompt: str,
    max_new_tokens: int = 256,
    temperature: float = 0.7,
    top_p: float = 0.9,
    top_k: int = 50,
    do_sample: bool = True,
    repetition_penalty: float = 1.1,
) -> str:
    """Sinh văn bản từ prompt và đo thời gian."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    start = time.perf_counter()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            do_sample=do_sample,
            repetition_penalty=repetition_penalty,
            pad_token_id=tokenizer.pad_token_id,
        )

    elapsed = time.perf_counter() - start
    generated_tokens = outputs.shape[1] - input_len

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print(f"⏱  Thời gian  : {elapsed:.2f}s")
    print(f"📝 Tokens sinh : {generated_tokens}")
    print(f"⚡ Tốc độ     : {generated_tokens / elapsed:.1f} tokens/s")
    print("-" * 60)

    return result

## 4. Test sinh văn bản cơ bản

In [ ]:
prompt_1 = "Artificial intelligence is"

result = generate_text(prompt_1, max_new_tokens=128)
print(result)

In [ ]:
prompt_2 = "The key benefits of using large language models in business are:"

result = generate_text(prompt_2, max_new_tokens=200)
print(result)

## 5. Test với prompt dạng Meeting Summary

Đây là task chính của dự án — kiểm tra xem base model xử lý prompt tóm tắt cuộc họp như thế nào **trước khi fine-tune**.

In [ ]:
meeting_transcript = """Meeting Transcript:
John: Good morning everyone. Let's start with the Q3 report.
Sarah: Revenue is up 15% compared to last quarter. We hit $2.3 million.
John: Great news. What about customer acquisition?
Mike: We onboarded 340 new customers. Churn rate dropped to 3.2%.
Sarah: Marketing spent $180K this quarter, mainly on digital campaigns.
John: ROI looks solid. Any concerns?
Mike: We need to invest more in customer support. Response times are increasing.
John: Agreed. Let's allocate budget for that next quarter.
Sarah: I'll prepare a proposal by Friday.
John: Perfect. Meeting adjourned.

Summary of this meeting:"""

result = generate_text(meeting_transcript, max_new_tokens=256)
print(result)

In [ ]:
# Test với prompt tiếng Việt
meeting_vn = """Biên bản cuộc họp:
Anh Minh: Chào mọi người, hôm nay chúng ta sẽ bàn về kế hoạch ra mắt sản phẩm mới.
Chị Lan: Đội thiết kế đã hoàn thành mockup. Cần review từ đội kỹ thuật.
Anh Tuấn: Đội dev có thể bắt đầu implement từ tuần sau nếu mockup được duyệt.
Anh Minh: Deadline ra mắt là cuối tháng 4. Chúng ta có kịp không?
Anh Tuấn: Nếu không có thay đổi lớn thì kịp. Cần thêm 1 backend developer.
Chị Lan: Tôi sẽ gửi mockup cho team review chiều nay.
Anh Minh: Ok, họp lại thứ 4 tuần sau để cập nhật tiến độ.

Tóm tắt cuộc họp:"""

result = generate_text(meeting_vn, max_new_tokens=256)
print(result)

## 6. Đo VRAM sử dụng

In [ ]:
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved() / 1e9
    total     = torch.cuda.get_device_properties(0).total_mem / 1e9

    print(f"VRAM Allocated : {allocated:.2f} GB")
    print(f"VRAM Reserved  : {reserved:.2f} GB")
    print(f"VRAM Total     : {total:.1f} GB")
    print(f"VRAM Free      : {total - reserved:.2f} GB")
else:
    print("Không có GPU — đang chạy trên CPU.")

## 7. So sánh Greedy vs Sampling

In [ ]:
test_prompt = "The most important thing in a meeting is"

print("=" * 60)
print("GREEDY (do_sample=False)")
print("=" * 60)
greedy = generate_text(test_prompt, max_new_tokens=100, do_sample=False)
print(greedy)

print("\n")

print("=" * 60)
print("SAMPLING (temperature=0.7, top_p=0.9)")
print("=" * 60)
sampled = generate_text(test_prompt, max_new_tokens=100, do_sample=True, temperature=0.7)
print(sampled)

## 8. Giải phóng bộ nhớ

In [ ]:
# Chạy cell này khi muốn giải phóng VRAM
import gc

del model
del tokenizer
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"VRAM sau giải phóng: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

print("Đã giải phóng bộ nhớ.")